# Experiential Learning Assignment – 2
## LLMOps: API Cost, Latency, and Prompt Version Management


In [ ]:
!pip install -q groq mlflow pandas numpy python-dotenv

# Import Libraries

In [ ]:
import os, json, time, mlflow
import pandas as pd
import numpy as np
from groq import Groq
from google.colab import userdata

In [ ]:
from groq import Groq
from google.colab import userdata

client = Groq(
    api_key=userdata.get("GROQ_API_KEY")
)

models = client.models.list()

for model in models.data:
    print(model.id)

openai/gpt-oss-120b
whisper-large-v3
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-arabic-saudi
whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-v1-english
meta-llama/llama-prompt-guard-2-86m
allam-2-7b
qwen/qwen3.8-27b
openai/gpt-oss-20b


# Configure Groq API

In [ ]:
api_key = userdata.get("GROQ_API_KEY")
client = Groq(api_key=api_key)
print("Groq Connected Successfully")

Groq Connected Successfully


# Model Configuration

In [ ]:
MODEL="meta-llama/llama-prompt-guard-2-22m"
INPUT_PRICE_PER_MILLION=0.05
OUTPUT_PRICE_PER_MILLION=0.08

# Questions Dataset

In [ ]:
questions=["What is machine learning?","Explain overfitting in simple terms.","What is gradient descent?","Differentiate supervised and unsupervised learning.","What is cross validation and why is it used?"]

# Prompt Versions

In [ ]:
PROMPT_V1="""Answer the following student question clearly.

Question: {question}"""
PROMPT_V2="""You are an academic assistant. Answer the following student question in a clear, concise and accurate manner. Use simple language suitable for undergraduate students.
Question: {question}"""
PROMPT_V3="""You are an academic assistant. Answer the student question and return ONLY valid JSON.
Use schema:{{"answer":"string","confidence":"high | medium | low","key_points":["string"]}}
Question: {question}"""

# Utility Functions

In [ ]:
def call_llm(prompt):
    start=time.time()
    response=client.chat.completions.create(model=MODEL,messages=[{"role":"user","content":prompt}],temperature=0)
    latency=time.time()-start
    return response, latency

def is_valid_json(text):
    try:
        json.loads(text)
        return 1
    except:
        return 0

# Task 1 - Latency Measurement

In [ ]:
latency_records=[]
for idx,question in enumerate(questions,1):
    prompt=PROMPT_V1.format(question=question)
    st=time.time()
    response=client.chat.completions.create(model=MODEL,messages=[{"role":"user","content":prompt}])
    et=time.time()
    latency_records.append({"question_id":idx,"prompt_version":"v1","model_name":MODEL,"start_time":st,"end_time":et,"latency_seconds":et-st,"status":"success"})
latency_df=pd.DataFrame(latency_records)
latency_df

,question_id,prompt_version,model_name,start_time,end_time,latency_seconds,status
0,1,v1,meta-llama/llama-prompt-guard-2-22m,1.793365e+09,1.793365e+09,0.166044,success
1,2,v1,meta-llama/llama-prompt-guard-2-22m,1.793365e+09,1.793365e+09,0.141797,success
2,3,v1,meta-llama/llama-prompt-guard-2-22m,1.793365e+09,1.793365e+09,0.124066,success
3,4,v1,meta-llama/llama-prompt-guard-2-22m,1.793365e+09,1.793365e+09,0.133080,success
4,5,v1,meta-llama/llama-prompt-guard-2-22m,1.793365e+09,1.793365e+09,0.129981,success


In [ ]:
latency_summary=pd.DataFrame({"Metric":["Mean","Median","Minimum","Maximum","Standard Deviation"],"Value":[latency_df["latency_seconds"].mean(),latency_df["latency_seconds"].median(),latency_df["latency_seconds"].min(),latency_df["latency_seconds"].max(),latency_df["latency_seconds"].std()]})
latency_summary

,Metric,Value
0,Mean,0.138994
1,Median,0.133080
2,Minimum,0.124066
3,Maximum,0.166044
4,Standard Deviation,0.016421


# Task 2 - Token Usage and Cost

In [ ]:
cost_records=[]
for idx,question in enumerate(questions,1):
    response,latency=call_llm(PROMPT_V1.format(question=question))
    usage=response.usage
    input_tokens=usage.prompt_tokens
    output_tokens=usage.completion_tokens
    total_tokens=usage.total_tokens
    total_cost=(input_tokens/1_000_000)*INPUT_PRICE_PER_MILLION+(output_tokens/1_000_000)*OUTPUT_PRICE_PER_MILLION
    cost_records.append({"question_id":idx,"input_tokens":input_tokens,"output_tokens":output_tokens,"total_tokens":total_tokens,"cost_usd":total_cost})
cost_df=pd.DataFrame(cost_records)
cost_df

,question_id,input_tokens,output_tokens,total_tokens,cost_usd
0,1,16,0,0,8.000000e-07
1,2,19,0,0,9.500000e-07
2,3,17,0,0,8.500000e-07
3,4,19,0,0,9.500000e-07
4,5,21,0,0,1.050000e-06


In [ ]:
monthly_cost=cost_df["cost_usd"].mean()*10000
print(monthly_cost)
combined=latency_df.merge(cost_df,on="question_id")
combined.to_csv("latency_tokens_cost.csv",index=False)

0.0092


# Task 3 - MLflow Evaluation

In [ ]:
mlflow.set_experiment("eduassist_llmops_assignment")
PROMPTS={"v1":PROMPT_V1,"v2":PROMPT_V2,"v3":PROMPT_V3}
results=[]
for version,template in PROMPTS.items():
    latencies=[];inputs=[];outputs=[];costs=[];json_scores=[]
    with mlflow.start_run(run_name=version):
        for question in questions:
            response,latency=call_llm(template.format(question=question))
            answer=response.choices[0].message.content
            usage=response.usage
            it=usage.prompt_tokens; ot=usage.completion_tokens
            cost=(it/1_000_000)*INPUT_PRICE_PER_MILLION+(ot/1_000_000)*OUTPUT_PRICE_PER_MILLION
            latencies.append(latency);inputs.append(it);outputs.append(ot);costs.append(cost)
            json_scores.append(is_valid_json(answer) if version=="v3" else 0)
        mlflow.log_metric("avg_latency_seconds",float(np.mean(latencies)))
        mlflow.log_metric("avg_input_tokens",float(np.mean(inputs)))
        mlflow.log_metric("avg_output_tokens",float(np.mean(outputs)))
        mlflow.log_metric("avg_cost_per_request",float(np.mean(costs)))
        mlflow.log_metric("valid_json",float(np.mean(json_scores)))
        results.append({"Prompt Version":version,"Avg Latency":np.mean(latencies),"Avg Input Tokens":np.mean(inputs),"Avg Output Tokens":np.mean(outputs),"Avg Cost":np.mean(costs),"Valid JSON Rate":np.mean(json_scores)})
comparison_df=pd.DataFrame(results)
comparison_df.to_csv("prompt_evaluation_results.csv",index=False)
comparison_df

,Prompt Version,Avg Latency,Avg Input Tokens,Avg Output Tokens,Avg Cost,Valid JSON Rate
0,v1,0.136680,18.4,0.0,9.200000e-07,0.0
1,v2,0.126572,41.4,0.0,2.070000e-06,0.0
2,v3,0.127412,63.4,0.0,3.170000e-06,1.0


In [ ]:
!mlflow ui --host 0.0.0.0 --port 5000

Backend store URI not provided. Using sqlite:///mlflow.db
Registry store URI not provided. Using backend store URI.
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
2026/10/30 13:07:21 INFO mlflow.server.jobs.executor_registry: Available backends after discovery: local
2026/10/30 13:07:21 INFO mlflow.server.jobs.executor_registry: Job executor backend 'local' passed requirements check
2026/10/30 13:07:22 INFO:     Uvicorn running on http://0.0.0.0:5000 (Press CTRL+C to quit)
2026/10/30 13:07:22 INFO:     Started parent process [10157]
/usr/local/lib/python3.13/dist-packages/mlflow/server/fastapi_app.py:23: StarletteDeprecationWarning: starlette.middleware.wsgi is deprecated and will be removed in a future release. Please refer to https://github.com/abersheeran/a2wsgi as a replacement.
  from starlette.middleware.wsgi import WSGIResponder, build